# Agent hallucination checks, step by step

Companion notebook for the article *How to detect hallucinations in AI agents in production*. It runs the same checks as `run.py` on four hand-written agent traces, one step at a time. Standard library only, no API keys, no model calls.

A teaching example by Vighnesh Suresh Babu. Not Blue Guardrails' product and not affiliated with it.

In [1]:
import json
from checks import check_tool_args, check_claims, check_trajectory, evidence, judge_prompt, llm_judge_stub, check_trace
from run import load_traces, results, table

traces = load_traces()
for name, t in traces.items():
    print(f"{name} ({len(t['spans'])} spans)\n    {t['_note']}\n")

01_clean (4 spans)
    Clean run. The answer only states facts from the get_order result. Every check should pass.

02_unsupported_claim (4 spans)
    Claim level. check_stock returned stock 0 in Leipzig with restock on 2026-12-15. The answer invents a stock count, a warehouse and a ship date.

03_wrong_tool_argument (4 spans)
    Tool-call level (the test trace from the article). The user asked about A-77; the model called get_order with A-78 and an include value outside the schema enum. The call failed and the answer reports success anyway.

04_built_on_failed_step (6 spans)
    Trajectory level. Amount, currency, order ID and refund timing are all supported by earlier context, so the claim check passes. But issue_refund timed out, so the confirmation is false. Only the trajectory check catches it.



## What a trace looks like

Each file in `traces/` is one trace: a list of spans with flattened attributes named after the [OpenTelemetry GenAI semantic conventions](https://github.com/open-telemetry/semantic-conventions-genai). `chat` spans carry `gen_ai.system_instructions`, `gen_ai.input.messages`, `gen_ai.output.messages` and `gen_ai.tool.definitions`. `execute_tool` spans carry `gen_ai.tool.name`, `gen_ai.tool.call.id`, `gen_ai.tool.call.arguments`, `gen_ai.tool.call.result` and, when the call failed, `error.type`.

Here is the tool-call trace from the article: the user asks about order A-77.

In [2]:
def show(name):
    for s in traces[name]['spans']:
        a = s['attributes']
        op = a['gen_ai.operation.name']
        what = ''
        if op == 'chat':
            part = a['gen_ai.output.messages'][0]['parts'][0]
            what = part.get('content') or f"{part['name']}({json.dumps(part['arguments'])})"
        elif op == 'execute_tool':
            what = f"error.type={a['error.type']}" if a.get('error.type') else json.dumps(a['gen_ai.tool.call.result'])
        print(f"{s['span_id']}  {s['name']:<28} {what}")

show('03_wrong_tool_argument')

abeae02e03365b31  invoke_agent order_support   
dd146fd1b3ea8c90  chat demo-model              get_order({"order_id": "A-78", "include": "tracking"})
fc280d2db63c9a6a  execute_tool get_order       error.type=OrderNotFoundError
233022537a66b809  chat demo-model              Your order A-77 is on its way. You can follow it with the tracking link in your account.


## Step 1: tool arguments

Every `tool_call` the model emits is validated against the JSON Schema in `gen_ai.tool.definitions` (a small subset: type, enum, required, properties, additionalProperties, items). Then provenance: an argument whose name ends in `id`, `number`, `email`, `path` or `sku` must appear in the system instructions, a user message or an earlier tool result. The model's own earlier output does not count.

In [3]:
def show_findings(check):
    for name, t in traces.items():
        hits = check(t['spans'])
        print(f'{name:<24} {"FLAG" if hits else "PASS"}')
        for f in hits:
            print(f'    {f.issue}  span={f.span}  value={f.value}\n    > {f.detail}')

show_findings(check_tool_args)

01_clean                 PASS
02_unsupported_claim     PASS
03_wrong_tool_argument   FLAG
    schema_violation  span=dd146fd1b3ea8c90  value=include=tracking
    > get_order: 'tracking' is not one of ['summary', 'shipping', 'items']
    ungrounded_argument  span=dd146fd1b3ea8c90  value=order_id=A-78
    > get_order: 'A-78' appears in no instruction, user message or tool result
04_built_on_failed_step  PASS


## Step 2: claims against what the agent saw

For each answer, the check pulls out dates, IDs, numbers and capitalised names, and looks for each one in the evidence for that step: system instructions, user messages and tool results. Here is the evidence behind the stock answer in trace 02, then the result for all traces.

In [4]:
stock_answer = [s for s in traces['02_unsupported_claim']['spans']
                if s['attributes']['gen_ai.operation.name'] == 'chat'][-1]
print('EVIDENCE:\n' + evidence(stock_answer))
print('\nANSWER:\n' + stock_answer['attributes']['gen_ai.output.messages'][0]['parts'][0]['content'] + '\n')
show_findings(check_claims)

EVIDENCE:
You are the support agent for a home goods shop. Answer only from tool results. Refunds reach the customer's card in 5 to 7 business days. Confirm a refund only after issue_refund succeeds.
Is the blue kettle K-200 in stock? I need it before December 1.
{"sku": "K-200", "name": "Blue kettle", "stock": 0, "warehouse": "Leipzig", "restock_date": "2026-12-15"}

ANSWER:
Yes, K-200 is in stock: 12 units at our Hamburg warehouse. It can ship on November 28.

01_clean                 PASS
02_unsupported_claim     FLAG
    unsupported_date  span=1e9ea827a9b6137d  value=November 28
    > date 'November 28' is in the answer but in nothing the agent saw; agent saw: 2026-12-15, December 1
    unsupported_number  span=1e9ea827a9b6137d  value=12
    > number '12' is in the answer but in nothing the agent saw; agent saw: 5, 7, 0
    unsupported_entity  span=1e9ea827a9b6137d  value=Hamburg
    > entity 'Hamburg' is in the answer but in nothing the agent saw
03_wrong_tool_argument   PASS
04_b

## Step 3: trajectory

Tool calls are linked to their results by `gen_ai.tool.call.id`. A call with `error.type` or an empty result is a failed step. An answer written after a failed step is flagged unless it says the step failed, or a later call of the same tool succeeded. Trace 04 shows why this level matters: every number in the refund confirmation is supported by earlier context, so the claim check passes, but the refund itself timed out.

In [5]:
show_findings(check_trajectory)

01_clean                 PASS
02_unsupported_claim     PASS
03_wrong_tool_argument   FLAG
    answer_after_failed_step  span=233022537a66b809  value=get_order
    > answer follows get_order call call_f3c729_1 (OrderNotFoundError, span fc280d2db63c9a6a) and does not say it failed
04_built_on_failed_step  FLAG
    answer_after_failed_step  span=182880851334387f  value=issue_refund
    > answer follows issue_refund call call_20cfea_2 (timeout, span b4505ac19d72d37b) and does not say it failed


## All checks, all traces

The same table `python run.py -v` prints.

In [6]:
print(table(results(traces), verbose=True))

TRACE                    CHECK       VERDICT  EVIDENCE SPAN     ISSUE                     VALUE
-----------------------  ----------  -------  ----------------  ------------------------  ----------------
01_clean                 tool_args   PASS     -                 -                         -
01_clean                 claims      PASS     -                 -                         -
01_clean                 trajectory  PASS     -                 -                         -
02_unsupported_claim     tool_args   PASS     -                 -                         -
02_unsupported_claim     claims      FLAG     1e9ea827a9b6137d  unsupported_date          November 28
                         > date 'November 28' is in the answer but in nothing the agent saw; agent saw: 2026-12-15, December 1
02_unsupported_claim     claims      FLAG     1e9ea827a9b6137d  unsupported_number        12
                         > number '12' is in the answer but in nothing the agent saw; agent saw: 5, 7, 0
02

## Optional: LLM judge (off by default)

Some errors have no number or name to match. In trace 02 the words "in stock" contradict `"stock": 0`, and only a verifier that reads meaning can say so. `check_trace(spans, judge=fn)` sends each answer with its evidence to `fn` as one prompt with a narrow rule, domain context and span-level output. The default is no judge. `llm_judge_stub` is a placeholder that raises until you wire in a model you have evaluated against human labels.

In [7]:
print(judge_prompt(stock_answer['attributes']['gen_ai.output.messages'][0]['parts'][0]['content'],
                   evidence(stock_answer)))

try:
    check_trace(traces['02_unsupported_claim']['spans'], judge=llm_judge_stub)
except NotImplementedError as e:
    print('\nllm_judge_stub:', e)

Domain: Customer support agent for a home goods shop. Sources: order, stock and refund tools.
Issue type: unsupported_claim: the answer states a fact (number, date, name, status or outcome) that conflicts with, or is missing from, the evidence the agent had at this step.

Evidence (everything the agent saw at this step):
You are the support agent for a home goods shop. Answer only from tool results. Refunds reach the customer's card in 5 to 7 business days. Confirm a refund only after issue_refund succeeds.
Is the blue kettle K-200 in stock? I need it before December 1.
{"sku": "K-200", "name": "Blue kettle", "stock": 0, "warehouse": "Leipzig", "restock_date": "2026-12-15"}

Answer to check:
Yes, K-200 is in stock: 12 units at our Hamburg warehouse. It can ship on November 28.

Return a JSON list: [{"text": "<exact flagged span>", "label": "unsupported_claim", "explanation": "<what the evidence says instead>"}], or [] if nothing breaks the rule.

llm_judge_stub: No LLM judge is wired u

## What this does not do

- It is not a detector you can deploy. The regexes are tuned to four synthetic traces and will raise false positives on real text (a greeting with the user's name, a number from a policy the prompt does not contain).
- It checks support against the agent's own context, not truth in the world.
- Paraphrase, inference, conflation and drift from the system prompt need an LLM verifier with domain context, and that verifier needs to be measured on labelled traces before its flags mean anything.